In [ ]:
%pip uninstall -y matplotlib matplotlib-inline
%pip install --no-cache-dir --force-reinstall "matplotlib==3.8.4" matplotlib-inline

In [ ]:
import os
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

import matplotlib.pyplot as plt

from pymatgen.core import Composition, Element

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance

from scipy.stats import spearmanr
import joblib

warnings.filterwarnings("ignore")

BASE_DIR = Path("sodium_cathode_ml_protocols")
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "outputs"
METRICS_DIR = OUTPUT_DIR / "metrics"
PRED_DIR = OUTPUT_DIR / "predictions"
FIG_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
FEATURE_DIR = OUTPUT_DIR / "features"
TABLE_DIR = OUTPUT_DIR / "tables"

for folder in [BASE_DIR, INPUT_DIR, OUTPUT_DIR, METRICS_DIR, PRED_DIR, FIG_DIR, MODEL_DIR, FEATURE_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print("Project folder:", BASE_DIR.resolve())

In [ ]:
def find_first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError(
        "None of these files were found:\n" + "\n".join(str(x) for x in candidates)
    )


MASTER_PATH = find_first_existing([
    Path("mp_na_electrodes_master_v1_with_cde_evidence.csv"),
    Path("sodium_cathode_cde_matching/outputs/mp_na_electrodes_master_v1_with_cde_evidence.csv"),
    Path("/mnt/data/mp_na_electrodes_master_v1_with_cde_evidence.csv"),
])

AUDIT_PATH = find_first_existing([
    Path("cde_matching_audit_summary.csv"),
    Path("sodium_cathode_cde_matching/audit/cde_matching_audit_summary.csv"),
    Path("/mnt/data/cde_matching_audit_summary.csv"),
])

TOP_EVIDENCE_PATH = find_first_existing([
    Path("top_conservative_candidates_with_cde_evidence.csv"),
    Path("sodium_cathode_cde_matching/outputs/top_conservative_candidates_with_cde_evidence.csv"),
    Path("/mnt/data/top_conservative_candidates_with_cde_evidence.csv"),
])

df = pd.read_csv(MASTER_PATH)
df_audit = pd.read_csv(AUDIT_PATH)
df_top_evidence = pd.read_csv(TOP_EVIDENCE_PATH)

print("Master dataset:", MASTER_PATH)
print("Shape:", df.shape)

print("\nAudit summary:")
display(df_audit)

print("\nTop CDE-supported conservative candidates:")
display(df_top_evidence.head(20))

In [ ]:
def to_bool_series(s):
    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
        .fillna(False)
    )


bool_cols = [
    "basic_screening_pass",
    "hard_exclusion_free_candidate",
    "moderate_sustainable_candidate",
    "conservative_earth_abundant_candidate",
    "cde_exact_formula_match",
    "cde_has_capacity_evidence",
    "cde_has_voltage_evidence",
    "cde_has_capacity_and_voltage_evidence",
]

for col in bool_cols:
    if col in df.columns:
        df[col] = to_bool_series(df[col])

numeric_cols = [
    "average_voltage",
    "capacity_grav",
    "capacity_vol",
    "energy_grav",
    "energy_vol",
    "max_delta_volume",
    "stability_charge",
    "stability_discharge",
    "fracA_charge",
    "fracA_discharge",
    "max_voltage_step",
    "mp_preliminary_score",
    "cde_capacity_median",
    "cde_voltage_median",
    "cde_energy_median",
    "cde_total_doi_count",
    "cde_capacity_record_count",
    "cde_voltage_record_count",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["stability_worst"] = df[["stability_charge", "stability_discharge"]].max(axis=1)

print("Rows:", len(df))
print("Conservative candidates:", int(df.get("conservative_earth_abundant_candidate", pd.Series(False, index=df.index)).sum()))
print("Exact CDE evidence:", int(df.get("cde_exact_formula_match", pd.Series(False, index=df.index)).sum()))

display(df.head())

In [ ]:
def formula_to_reduced(formula):
    if pd.isna(formula):
        return None
    try:
        return Composition(str(formula)).reduced_formula
    except Exception:
        return None


def choose_model_formula(row):
    """
    Use discharge formula first because it is usually a normal full Na-containing formula.
    Avoid battery_formula like Na0-2MnPO4F for descriptor generation.
    """
    for col in ["formula_discharge", "formula_for_parsing", "framework_formula", "formula_charge"]:
        if col in row.index:
            value = row.get(col)
            if pd.notna(value) and str(value).strip():
                try:
                    Composition(str(value))
                    return str(value)
                except Exception:
                    pass
    return None


df["ml_formula"] = df.apply(choose_model_formula, axis=1)
df["ml_reduced_formula"] = df["ml_formula"].map(formula_to_reduced)

failed = df["ml_reduced_formula"].isna().sum()

print("Formula parsing failed rows:", failed)
display(df[["battery_formula", "formula_discharge", "framework_formula", "ml_formula", "ml_reduced_formula"]].head(20))

In [ ]:
MAIN_ELEMENTS = [
    "Na", "O", "P", "F", "S", "C", "N", "Si",
    "Fe", "Mn", "Ti", "Cu", "V", "Cr", "Co", "Ni",
    "Mo", "W", "Al", "Mg", "Ca", "K"
]

ELEMENT_PROPS = [
    "Z",
    "atomic_mass",
    "X",
    "row",
    "group",
    "atomic_radius",
]


def safe_element_prop(el_symbol, prop):
    try:
        el = Element(el_symbol)

        if prop == "Z":
            return float(el.Z)

        if prop == "atomic_mass":
            return float(el.atomic_mass)

        if prop == "X":
            return float(el.X) if el.X is not None else np.nan

        if prop == "row":
            return float(el.row)

        if prop == "group":
            return float(el.group)

        if prop == "atomic_radius":
            return float(el.atomic_radius) if el.atomic_radius is not None else np.nan

    except Exception:
        return np.nan

    return np.nan


def weighted_stats(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)

    if mask.sum() == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "max": np.nan,
            "range": np.nan,
        }

    values = values[mask]
    weights = weights[mask] / weights[mask].sum()

    mean = np.sum(values * weights)
    var = np.sum(weights * (values - mean) ** 2)

    return {
        "mean": mean,
        "std": math.sqrt(var),
        "min": np.min(values),
        "max": np.max(values),
        "range": np.max(values) - np.min(values),
    }


def composition_descriptors(formula):
    features = {}

    if pd.isna(formula):
        return features

    try:
        comp = Composition(str(formula))
        el_amt = comp.get_el_amt_dict()
        total_atoms = sum(el_amt.values())
        elements = list(el_amt.keys())
        weights = np.array([el_amt[e] / total_atoms for e in elements], dtype=float)

        features["comp__total_atoms"] = total_atoms
        features["comp__n_elements"] = len(elements)

        for e in MAIN_ELEMENTS:
            features[f"comp__frac_{e}"] = el_amt.get(e, 0.0) / total_atoms

        transition_metals = {"Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "Mo", "W"}
        anions = {"O", "F", "S", "N", "C"}

        features["comp__transition_metal_frac"] = sum(
            el_amt.get(e, 0.0) for e in transition_metals
        ) / total_atoms

        features["comp__anion_frac"] = sum(
            el_amt.get(e, 0.0) for e in anions
        ) / total_atoms

        features["comp__polyanion_indicator_PO"] = float(("P" in elements) and ("O" in elements))
        features["comp__oxide_indicator"] = float("O" in elements)
        features["comp__fluoride_indicator"] = float("F" in elements)
        features["comp__sulfate_like_indicator"] = float(("S" in elements) and ("O" in elements))
        features["comp__carbon_indicator"] = float("C" in elements)

        for prop in ELEMENT_PROPS:
            vals = [safe_element_prop(e, prop) for e in elements]
            stats = weighted_stats(vals, weights)

            for stat_name, stat_value in stats.items():
                features[f"comp__{prop}_{stat_name}"] = stat_value

    except Exception:
        pass

    return features


descriptor_rows = [composition_descriptors(f) for f in tqdm(df["ml_formula"], desc="Composition descriptors")]
df_comp = pd.DataFrame(descriptor_rows)

print("Composition descriptor shape:", df_comp.shape)
display(df_comp.head())

comp_path = FEATURE_DIR / "protocol_A_composition_descriptors.csv"
df_comp.to_csv(comp_path, index=False)
print("Saved:", comp_path)

In [ ]:
df_features = pd.concat(
    [
        df.reset_index(drop=True),
        df_comp.reset_index(drop=True),
    ],
    axis=1,
)

PROTOCOL_A_FEATURES = [c for c in df_features.columns if c.startswith("comp__")]

# Protocol B: composition + simple structural/lattice descriptors
PROTOCOL_B_EXTRA = [
    "host_structure__lattice__a",
    "host_structure__lattice__b",
    "host_structure__lattice__c",
    "host_structure__lattice__alpha",
    "host_structure__lattice__beta",
    "host_structure__lattice__gamma",
    "host_structure__lattice__volume",
]

PROTOCOL_B_EXTRA = [c for c in PROTOCOL_B_EXTRA if c in df_features.columns]
PROTOCOL_B_FEATURES = PROTOCOL_A_FEATURES + PROTOCOL_B_EXTRA

# Protocol C-strict: post-DFT electrode descriptors but no direct target columns, no CDE, no scores
PROTOCOL_C_EXTRA = [
    "fracA_charge",
    "fracA_discharge",
    "max_delta_volume",
    "stability_charge",
    "stability_discharge",
]

PROTOCOL_C_EXTRA = [c for c in PROTOCOL_C_EXTRA if c in df_features.columns]
PROTOCOL_C_FEATURES = PROTOCOL_B_FEATURES + PROTOCOL_C_EXTRA

feature_manifest = {
    "Protocol_A_composition_only": PROTOCOL_A_FEATURES,
    "Protocol_B_composition_plus_lattice": PROTOCOL_B_FEATURES,
    "Protocol_C_strict_post_DFT": PROTOCOL_C_FEATURES,
    "excluded_from_main_ML": [
        "average_voltage",
        "capacity_grav",
        "capacity_vol",
        "energy_grav",
        "energy_vol",
        "max_voltage_step",
        "mp_preliminary_score",
        "cde_capacity_median",
        "cde_voltage_median",
        "cde_energy_median",
        "cde_total_doi_count",
        "candidate labels",
        "screening scores",
    ],
}

manifest_path = FEATURE_DIR / "protocol_feature_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(feature_manifest, f, indent=2)

print("Protocol A features:", len(PROTOCOL_A_FEATURES))
print("Protocol B features:", len(PROTOCOL_B_FEATURES))
print("Protocol C-strict features:", len(PROTOCOL_C_FEATURES))
print("Saved:", manifest_path)

In [ ]:
TARGET_CONFIG = {
    "average_voltage": {
        "higher_is_better": True,
        "unit": "V",
    },
    "capacity_grav": {
        "higher_is_better": True,
        "unit": "mAh/g",
    },
    "energy_grav": {
        "higher_is_better": True,
        "unit": "Wh/kg",
    },
    "stability_worst": {
        "higher_is_better": False,
        "unit": "eV/atom",
    },
}

# Group by framework formula to reduce leakage from near-duplicate electrode entries.
if "framework_formula" in df_features.columns:
    df_features["cv_group"] = df_features["framework_formula"].fillna(df_features["ml_reduced_formula"]).astype(str)
else:
    df_features["cv_group"] = df_features["ml_reduced_formula"].astype(str)

print("Unique CV groups:", df_features["cv_group"].nunique())
display(df_features[["battery_formula", "formula_discharge", "framework_formula", "ml_reduced_formula", "cv_group"]].head(20))

In [ ]:
MODELS = {
    "Ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ]),

    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=500,
            random_state=RANDOM_STATE,
            min_samples_leaf=2,
            n_jobs=-1,
        )),
    ]),

    "ExtraTrees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=500,
            random_state=RANDOM_STATE,
            min_samples_leaf=2,
            n_jobs=-1,
        )),
    ]),

    "GradientBoosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(
            random_state=RANDOM_STATE,
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
        )),
    ]),
}

print("Models:", list(MODELS.keys()))

In [ ]:
# Fixed Cell 9A — Create/check unique row ID for safe ML merging

if "mp_index" not in df_features.columns:
    df_features["mp_index"] = np.arange(len(df_features))

if "mp_index" not in df.columns:
    df["mp_index"] = np.arange(len(df))

df_features["mp_index"] = pd.to_numeric(df_features["mp_index"], errors="coerce").astype(int)
df["mp_index"] = pd.to_numeric(df["mp_index"], errors="coerce").astype(int)

assert df_features["mp_index"].is_unique, "ERROR: df_features mp_index is not unique."
assert df["mp_index"].is_unique, "ERROR: original df mp_index is not unique."

print("Row-ID check passed.")
print("df_features rows:", len(df_features))
print("original df rows:", len(df))
print("unique mp_index in df_features:", df_features["mp_index"].nunique())
print("unique mp_index in original df:", df["mp_index"].nunique())

display(df_features[["mp_index", "battery_formula", "formula_discharge", "framework_formula"]].head())


In [ ]:
# Fixed Cell 10 — Cross-validation functions with mp_index preserved

def clean_feature_list_for_target(feature_cols, target_name):
    """
    Remove target-leaking fields for each target/protocol.
    CDE evidence and candidate labels are never allowed as main ML features.
    """
    feature_cols = list(feature_cols)
    remove = set()

    if target_name == "stability_worst":
        remove.update(["stability_charge", "stability_discharge"])

    remove.add(target_name)

    forbidden_prefixes = ["cde_", "valid_", "score"]
    forbidden_exact = {
        "average_voltage",
        "capacity_grav",
        "capacity_vol",
        "energy_grav",
        "energy_vol",
        "max_voltage_step",
        "mp_preliminary_score",
        "voltage_score",
        "capacity_score",
        "energy_score",
        "volume_score",
        "stability_score",
        "criticality_score",
        "basic_screening_pass",
        "hard_exclusion_free_candidate",
        "moderate_sustainable_candidate",
        "conservative_earth_abundant_candidate",
        "cde_exact_formula_match",
        "cde_has_capacity_evidence",
        "cde_has_voltage_evidence",
        "cde_has_capacity_and_voltage_evidence",
    }

    safe_cols = []

    for c in feature_cols:
        if c in remove:
            continue
        if c in forbidden_exact:
            continue
        if any(str(c).startswith(prefix) for prefix in forbidden_prefixes):
            continue
        safe_cols.append(c)

    return safe_cols


def make_cv_splitter(y, groups, n_splits=5):
    """
    Use GroupKFold when framework groups exist.
    Fallback to KFold only if grouping is impossible.
    """
    groups = pd.Series(groups).astype(str).fillna("missing_group")
    unique_groups = groups.nunique()

    if unique_groups >= 2:
        k = min(n_splits, unique_groups)
        return GroupKFold(n_splits=k), groups

    k = min(n_splits, len(y))
    return KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE), None


def top_fraction_enrichment(y_true, y_pred, higher_is_better=True, frac=0.20):
    """
    Measures whether the model retrieves the true top-performing candidates.
    Enrichment > 1 is better than random.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    n = len(y_true)

    if n < 5:
        return np.nan

    k = max(1, int(np.ceil(frac * n)))

    if higher_is_better:
        true_top = set(np.argsort(y_true)[-k:])
        pred_top = set(np.argsort(y_pred)[-k:])
    else:
        true_top = set(np.argsort(y_true)[:k])
        pred_top = set(np.argsort(y_pred)[:k])

    observed_overlap = len(true_top & pred_top)
    expected_overlap = k * k / n

    if expected_overlap == 0:
        return np.nan

    return observed_overlap / expected_overlap


def evaluate_predictions(y_true, y_pred, higher_is_better=True):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    try:
        r2 = r2_score(y_true, y_pred)
    except Exception:
        r2 = np.nan

    try:
        spearman = spearmanr(y_true, y_pred).correlation
    except Exception:
        spearman = np.nan

    enrichment_20 = top_fraction_enrichment(
        y_true,
        y_pred,
        higher_is_better=higher_is_better,
        frac=0.20,
    )

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "spearman": spearman,
        "top20_enrichment": enrichment_20,
    }


def run_grouped_cv(df_input, feature_cols, target_name, protocol_name, model_name, model):
    """
    Run grouped cross-validation and preserve mp_index for safe later merging.
    """
    feature_cols = clean_feature_list_for_target(feature_cols, target_name)

    required_meta_cols = [
        "mp_index",
        "cv_group",
        "battery_formula",
        "formula_discharge",
        "framework_formula",
        "preliminary_family",
    ]

    available_meta_cols = [
        c for c in required_meta_cols
        if c in df_input.columns
    ]

    if "mp_index" not in available_meta_cols:
        raise ValueError("mp_index missing. Run Fixed Cell 9A first.")

    if "cv_group" not in available_meta_cols:
        raise ValueError("cv_group missing. Run original Cell 8 first.")

    model_df = df_input[
        feature_cols + [target_name] + available_meta_cols
    ].copy()

    model_df[target_name] = pd.to_numeric(model_df[target_name], errors="coerce")
    model_df = model_df[model_df[target_name].notna()].reset_index(drop=True)

    X = model_df[feature_cols]
    y = model_df[target_name].values
    groups = model_df["cv_group"]

    splitter, split_groups = make_cv_splitter(y, groups, n_splits=5)

    oof_pred = np.full(len(model_df), np.nan)

    for fold, (train_idx, test_idx) in enumerate(
        splitter.split(X, y, groups=split_groups),
        start=1,
    ):
        est = clone(model)
        est.fit(X.iloc[train_idx], y[train_idx])
        oof_pred[test_idx] = est.predict(X.iloc[test_idx])

    higher_is_better = TARGET_CONFIG[target_name]["higher_is_better"]
    metrics = evaluate_predictions(
        y,
        oof_pred,
        higher_is_better=higher_is_better,
    )

    baseline_pred = np.full_like(y, np.nanmedian(y), dtype=float)
    baseline_metrics = evaluate_predictions(
        y,
        baseline_pred,
        higher_is_better=higher_is_better,
    )

    metric_row = {
        "target": target_name,
        "unit": TARGET_CONFIG[target_name]["unit"],
        "protocol": protocol_name,
        "model": model_name,
        "n_samples": len(model_df),
        "n_features": len(feature_cols),
        "n_groups": groups.nunique(),
        **metrics,
        "baseline_rmse": baseline_metrics["rmse"],
        "baseline_mae": baseline_metrics["mae"],
    }

    pred_cols = [
        "mp_index",
        "battery_formula",
        "formula_discharge",
        "framework_formula",
        "preliminary_family",
        "cv_group",
        target_name,
    ]

    pred_cols = [
        c for c in pred_cols
        if c in model_df.columns
    ]

    pred_df = model_df[pred_cols].copy()
    pred_df["target"] = target_name
    pred_df["protocol"] = protocol_name
    pred_df["model"] = model_name
    pred_df["y_true"] = y
    pred_df["y_pred_oof"] = oof_pred
    pred_df["abs_error"] = np.abs(pred_df["y_true"] - pred_df["y_pred_oof"])

    return metric_row, pred_df, feature_cols


print("Fixed CV functions loaded successfully.")


In [ ]:
# Fixed Cell 11 — Run grouped CV for all targets, protocols, and models

PROTOCOLS = {
    "A_composition_only": PROTOCOL_A_FEATURES,
    "B_composition_lattice": PROTOCOL_B_FEATURES,
    "C_strict_post_DFT": PROTOCOL_C_FEATURES,
}

all_metrics = []
all_predictions = []
used_feature_rows = []

for target_name in TARGET_CONFIG.keys():
    for protocol_name, feature_cols in PROTOCOLS.items():
        for model_name, model in MODELS.items():

            print(f"Running: target={target_name}, protocol={protocol_name}, model={model_name}")

            metric_row, pred_df, used_cols = run_grouped_cv(
                df_input=df_features,
                feature_cols=feature_cols,
                target_name=target_name,
                protocol_name=protocol_name,
                model_name=model_name,
                model=model,
            )

            all_metrics.append(metric_row)
            all_predictions.append(pred_df)

            used_feature_rows.append({
                "target": target_name,
                "protocol": protocol_name,
                "model": model_name,
                "n_used_features": len(used_cols),
                "used_features": json.dumps(used_cols),
            })

df_metrics = pd.DataFrame(all_metrics)
df_predictions = pd.concat(all_predictions, ignore_index=True)
df_used_features = pd.DataFrame(used_feature_rows)

metrics_path = METRICS_DIR / "ml_grouped_cv_metrics_all_targets.csv"
pred_path = PRED_DIR / "ml_grouped_cv_oof_predictions_all_targets.csv"
used_features_path = FEATURE_DIR / "ml_used_features_by_target_protocol_model.csv"

df_metrics.to_csv(metrics_path, index=False)
df_predictions.to_csv(pred_path, index=False)
df_used_features.to_csv(used_features_path, index=False)

print("Saved:", metrics_path)
print("Saved:", pred_path)
print("Saved:", used_features_path)

print("Prediction rows:", len(df_predictions))
print("Unique mp_index in predictions:", df_predictions["mp_index"].nunique())

display(
    df_metrics.sort_values(["target", "rmse"])
)


In [ ]:
df_best = (
    df_metrics
    .sort_values(["target", "protocol", "rmse"], ascending=[True, True, True])
    .groupby(["target", "protocol"], as_index=False)
    .first()
)

best_path = METRICS_DIR / "ml_best_model_by_target_protocol.csv"
df_best.to_csv(best_path, index=False)

print("Saved:", best_path)
display(df_best)

In [ ]:
summary_cols = [
    "target",
    "protocol",
    "model",
    "n_samples",
    "n_features",
    "n_groups",
    "rmse",
    "mae",
    "r2",
    "spearman",
    "top20_enrichment",
    "baseline_rmse",
    "baseline_mae",
]

df_summary = df_best[summary_cols].copy()

for col in ["rmse", "mae", "r2", "spearman", "top20_enrichment", "baseline_rmse", "baseline_mae"]:
    df_summary[col] = pd.to_numeric(df_summary[col], errors="coerce").round(4)

summary_path = TABLE_DIR / "table_ml_grouped_cv_summary.csv"
df_summary.to_csv(summary_path, index=False)

print("Saved:", summary_path)
display(df_summary)

In [ ]:
# Safe Cell 14 — Parity plot data + optional plots

parity_data_dir = FIG_DIR / "parity_plot_data"
parity_data_dir.mkdir(parents=True, exist_ok=True)

for target_name in TARGET_CONFIG.keys():
    row = df_best[
        (df_best["target"] == target_name)
        & (df_best["protocol"] == "C_strict_post_DFT")
    ]

    if row.empty:
        continue

    best_model_name = row.iloc[0]["model"]

    plot_df = df_predictions[
        (df_predictions["target"] == target_name)
        & (df_predictions["protocol"] == "C_strict_post_DFT")
        & (df_predictions["model"] == best_model_name)
    ].copy()

    if plot_df.empty:
        continue

    # Always save plot data, even if plotting fails
    parity_csv = parity_data_dir / f"parity_data_{target_name}_C_strict_{best_model_name}.csv"
    plot_df.to_csv(parity_csv, index=False)
    print("Saved parity data:", parity_csv)

    try:
        import matplotlib
        matplotlib.rcParams["font.enable_last_resort"] = False
        matplotlib.rcParams["font.family"] = "DejaVu Sans"

        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(5, 5))

        ax.scatter(plot_df["y_true"], plot_df["y_pred_oof"], alpha=0.7)

        min_val = np.nanmin([plot_df["y_true"].min(), plot_df["y_pred_oof"].min()])
        max_val = np.nanmax([plot_df["y_true"].max(), plot_df["y_pred_oof"].max()])

        ax.plot([min_val, max_val], [min_val, max_val], linestyle="--")

        ax.set_xlabel(f"Computed MP {target_name}")
        ax.set_ylabel(f"Grouped-CV predicted {target_name}")
        ax.set_title(f"{target_name} | C-strict | {best_model_name}")

        # Avoid tight_layout because your error occurred there
        fig.subplots_adjust(left=0.18, right=0.95, bottom=0.15, top=0.88)

        out_plot = FIG_DIR / f"parity_{target_name}_C_strict_{best_model_name}.png"
        fig.savefig(out_plot, dpi=300, bbox_inches="standard")
        plt.close(fig)

        print("Saved plot:", out_plot)

    except Exception as e:
        print(f"Plot skipped for {target_name} because Matplotlib failed.")
        print("Error:", e)

In [ ]:
importance_rows = []

for target_name in TARGET_CONFIG.keys():
    feature_cols = clean_feature_list_for_target(PROTOCOL_C_FEATURES, target_name)

    model_df = df_features[feature_cols + [target_name]].copy()
    model_df[target_name] = pd.to_numeric(model_df[target_name], errors="coerce")
    model_df = model_df[model_df[target_name].notna()].reset_index(drop=True)

    X = model_df[feature_cols]
    y = model_df[target_name].values

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=700,
            random_state=RANDOM_STATE,
            min_samples_leaf=2,
            n_jobs=-1,
        )),
    ])

    model.fit(X, y)

    fitted_tree = model.named_steps["model"]
    importances = fitted_tree.feature_importances_

    for feat, imp in zip(feature_cols, importances):
        importance_rows.append({
            "target": target_name,
            "protocol": "C_strict_post_DFT",
            "model": "ExtraTrees",
            "feature": feat,
            "importance": imp,
        })

    model_path = MODEL_DIR / f"extratrees_C_strict_{target_name}.joblib"
    joblib.dump(model, model_path)

    print("Saved model:", model_path)

df_importance = pd.DataFrame(importance_rows)
importance_path = FEATURE_DIR / "extratrees_C_strict_feature_importance.csv"
df_importance.to_csv(importance_path, index=False)

print("Saved:", importance_path)

for target_name in TARGET_CONFIG.keys():
    print("\nTop features for:", target_name)
    display(
        df_importance[df_importance["target"] == target_name]
        .sort_values("importance", ascending=False)
        .head(20)
    )

In [ ]:
# Safe Cell 16 — Feature-importance plot data + optional plots

importance_plot_data_dir = FIG_DIR / "feature_importance_plot_data"
importance_plot_data_dir.mkdir(parents=True, exist_ok=True)

for target_name in TARGET_CONFIG.keys():
    tmp = (
        df_importance[df_importance["target"] == target_name]
        .sort_values("importance", ascending=False)
        .head(20)
        .sort_values("importance", ascending=True)
    )

    if tmp.empty:
        continue

    importance_csv = importance_plot_data_dir / f"feature_importance_top20_{target_name}.csv"
    tmp.to_csv(importance_csv, index=False)
    print("Saved feature-importance data:", importance_csv)

    try:
        import matplotlib
        matplotlib.rcParams["font.enable_last_resort"] = False
        matplotlib.rcParams["font.family"] = "DejaVu Sans"

        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.barh(tmp["feature"], tmp["importance"])
        ax.set_xlabel("ExtraTrees feature importance")
        ax.set_title(f"Top C-strict features: {target_name}")

        # Avoid tight_layout
        fig.subplots_adjust(left=0.42, right=0.95, bottom=0.12, top=0.90)

        out_plot = FIG_DIR / f"feature_importance_C_strict_{target_name}.png"
        fig.savefig(out_plot, dpi=300, bbox_inches="standard")
        plt.close(fig)

        print("Saved plot:", out_plot)

    except Exception as e:
        print(f"Feature-importance plot skipped for {target_name} because Matplotlib failed.")
        print("Error:", e)

In [ ]:
# Fixed Cell 17 — Merge best C-strict OOF predictions back to master using mp_index

df_master_ml = df.copy()

if "mp_index" not in df_master_ml.columns:
    raise ValueError("mp_index is missing from df. Run Fixed Cell 9A first.")

if "mp_index" not in df_predictions.columns:
    raise ValueError("mp_index is missing from df_predictions. Replace Cell 10 and rerun Cells 10–13 first.")

df_master_ml["mp_index"] = pd.to_numeric(
    df_master_ml["mp_index"],
    errors="coerce"
).astype(int)

df_predictions["mp_index"] = pd.to_numeric(
    df_predictions["mp_index"],
    errors="coerce"
).astype(int)

assert df_master_ml["mp_index"].is_unique, "ERROR: Original master df has duplicate mp_index."

print("Original master rows:", len(df_master_ml))
print("Unique mp_index in master:", df_master_ml["mp_index"].nunique())

for target_name in TARGET_CONFIG.keys():
    row = df_best[
        (df_best["target"] == target_name)
        & (df_best["protocol"] == "C_strict_post_DFT")
    ]

    if row.empty:
        print("No C-strict best model found for:", target_name)
        continue

    best_model_name = row.iloc[0]["model"]

    pred_df = df_predictions[
        (df_predictions["target"] == target_name)
        & (df_predictions["protocol"] == "C_strict_post_DFT")
        & (df_predictions["model"] == best_model_name)
    ][
        ["mp_index", "y_pred_oof", "abs_error"]
    ].copy()

    pred_df["mp_index"] = pd.to_numeric(
        pred_df["mp_index"],
        errors="coerce"
    ).astype(int)

    pred_df = pred_df.drop_duplicates(subset=["mp_index"], keep="first")

    assert pred_df["mp_index"].is_unique, f"ERROR: Duplicate mp_index in predictions for {target_name}"

    pred_df = pred_df.rename(columns={
        "y_pred_oof": f"ml_oof_pred_{target_name}",
        "abs_error": f"ml_oof_abs_error_{target_name}",
    })

    df_master_ml = df_master_ml.merge(
        pred_df,
        on="mp_index",
        how="left",
        validate="one_to_one"
    )

    print(
        f"Merged {target_name} predictions using best C-strict model:",
        best_model_name
    )

assert len(df_master_ml) == len(df), (
    f"ERROR: Row count changed after merging predictions: {len(df)} -> {len(df_master_ml)}"
)

master_ml_path = OUTPUT_DIR / "mp_na_electrodes_master_v2_with_ml_predictions_FIXED.csv"
df_master_ml.to_csv(master_ml_path, index=False)

print("\nSaved:", master_ml_path)
print("Fixed master shape:", df_master_ml.shape)
print("Expected rows:", len(df))

display(df_master_ml.head())


In [ ]:
# Fixed Cell 18 — Candidate shortlist with no row inflation

df_shortlist = df_master_ml.copy()

def clip01(x):
    return np.clip(x, 0, 1)


def safe_bool_series(s, index):
    """
    Convert mixed boolean/string/object column to clean Boolean Series.
    """
    if isinstance(s, pd.Series):
        return (
            s.astype(str)
            .str.strip()
            .str.lower()
            .map({
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            })
            .fillna(False)
        )

    return pd.Series(False, index=index)


numeric_for_shortlist = [
    "cde_total_doi_count",
    "mp_preliminary_score",
    "average_voltage",
    "capacity_grav",
    "energy_grav",
    "max_delta_volume",
    "stability_charge",
    "stability_discharge",
]

for col in numeric_for_shortlist:
    if col in df_shortlist.columns:
        df_shortlist[col] = pd.to_numeric(df_shortlist[col], errors="coerce")

if "cde_total_doi_count" in df_shortlist.columns:
    doi_score = np.log1p(df_shortlist["cde_total_doi_count"].fillna(0))
    denom = np.log1p(max(1, df_shortlist["cde_total_doi_count"].fillna(0).max()))
    df_shortlist["literature_support_score"] = clip01(doi_score / denom)
else:
    df_shortlist["literature_support_score"] = 0.0

conservative_bool = safe_bool_series(
    df_shortlist["conservative_earth_abundant_candidate"]
    if "conservative_earth_abundant_candidate" in df_shortlist.columns
    else None,
    df_shortlist.index
)

capacity_voltage_bool = safe_bool_series(
    df_shortlist["cde_has_capacity_and_voltage_evidence"]
    if "cde_has_capacity_and_voltage_evidence" in df_shortlist.columns
    else None,
    df_shortlist.index
)

basic_bool = safe_bool_series(
    df_shortlist["basic_screening_pass"]
    if "basic_screening_pass" in df_shortlist.columns
    else None,
    df_shortlist.index
)

hard_free_bool = safe_bool_series(
    df_shortlist["hard_exclusion_free_candidate"]
    if "hard_exclusion_free_candidate" in df_shortlist.columns
    else None,
    df_shortlist.index
)

cde_exact_bool = safe_bool_series(
    df_shortlist["cde_exact_formula_match"]
    if "cde_exact_formula_match" in df_shortlist.columns
    else None,
    df_shortlist.index
)

df_shortlist["conservative_earth_abundant_candidate_clean"] = conservative_bool
df_shortlist["hard_exclusion_free_candidate_clean"] = hard_free_bool
df_shortlist["basic_screening_pass_clean"] = basic_bool
df_shortlist["cde_exact_formula_match_clean"] = cde_exact_bool

df_shortlist["conservative_bonus"] = conservative_bool.astype(float)
df_shortlist["capacity_voltage_evidence_bonus"] = capacity_voltage_bool.astype(float)

if "mp_preliminary_score" not in df_shortlist.columns:
    df_shortlist["mp_preliminary_score"] = 0.0

df_shortlist["final_decision_support_score"] = (
    0.65 * df_shortlist["mp_preliminary_score"].fillna(0)
    + 0.20 * df_shortlist["literature_support_score"].fillna(0)
    + 0.10 * df_shortlist["conservative_bonus"].fillna(0)
    + 0.05 * df_shortlist["capacity_voltage_evidence_bonus"].fillna(0)
)

candidate_mask = basic_bool & hard_free_bool
df_candidate_shortlist = df_shortlist[candidate_mask].copy()

df_candidate_shortlist = df_candidate_shortlist.sort_values(
    by=[
        "conservative_earth_abundant_candidate_clean",
        "cde_exact_formula_match_clean",
        "final_decision_support_score",
    ],
    ascending=[False, False, False]
)

shortlist_cols = [
    "mp_index",
    "battery_formula",
    "formula_discharge",
    "framework_formula",
    "preliminary_family",
    "average_voltage",
    "capacity_grav",
    "energy_grav",
    "max_delta_volume",
    "stability_charge",
    "stability_discharge",
    "mp_preliminary_score",
    "conservative_earth_abundant_candidate",
    "hard_exclusion_free_candidate",
    "basic_screening_pass",
    "cde_exact_formula_match",
    "cde_matched_reduced_formulas",
    "cde_capacity_median",
    "cde_voltage_median",
    "cde_energy_median",
    "cde_capacity_record_count",
    "cde_voltage_record_count",
    "cde_energy_record_count",
    "cde_total_doi_count",
    "literature_support_score",
    "final_decision_support_score",
]

for target_name in TARGET_CONFIG.keys():
    for prefix in ["ml_oof_pred_", "ml_oof_abs_error_"]:
        col = f"{prefix}{target_name}"
        if col in df_candidate_shortlist.columns:
            shortlist_cols.append(col)

shortlist_cols = [
    c for c in shortlist_cols
    if c in df_candidate_shortlist.columns
]

shortlist_all_path = TABLE_DIR / "candidate_shortlist_all_records_FIXED.csv"
df_candidate_shortlist[shortlist_cols].to_csv(shortlist_all_path, index=False)

dedup_keys = [
    c for c in ["formula_discharge", "framework_formula"]
    if c in df_candidate_shortlist.columns
]

if dedup_keys:
    df_candidate_shortlist_unique = (
        df_candidate_shortlist
        .sort_values(
            by=[
                "conservative_earth_abundant_candidate_clean",
                "cde_exact_formula_match_clean",
                "final_decision_support_score",
            ],
            ascending=[False, False, False]
        )
        .drop_duplicates(subset=dedup_keys, keep="first")
        .copy()
    )
else:
    df_candidate_shortlist_unique = df_candidate_shortlist.copy()

shortlist_unique_path = TABLE_DIR / "candidate_shortlist_unique_for_manuscript_FIXED.csv"
df_candidate_shortlist_unique[shortlist_cols].to_csv(shortlist_unique_path, index=False)

print("Saved:", shortlist_all_path)
print("Saved:", shortlist_unique_path)

print("\nRow-count checks:")
print("Original master rows:", len(df_master_ml))
print("All candidate shortlist rows:", len(df_candidate_shortlist))
print("Unique manuscript shortlist rows:", len(df_candidate_shortlist_unique))

assert len(df_candidate_shortlist) <= len(df_master_ml), "ERROR: Candidate shortlist is inflated."

display(df_candidate_shortlist_unique[shortlist_cols].head(50))


In [ ]:
# Fixed Cell 19 — Family-level summary without inflated rows

family_df = df_master_ml.copy()

for col in [
    "basic_screening_pass",
    "hard_exclusion_free_candidate",
    "conservative_earth_abundant_candidate",
    "cde_exact_formula_match",
]:
    if col in family_df.columns:
        family_df[col] = (
            family_df[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            })
            .fillna(False)
        )
    else:
        family_df[col] = False

for col in [
    "average_voltage",
    "capacity_grav",
    "energy_grav",
    "mp_preliminary_score",
]:
    if col in family_df.columns:
        family_df[col] = pd.to_numeric(family_df[col], errors="coerce")

family_summary = (
    family_df
    .groupby("preliminary_family", dropna=False)
    .agg(
        n_records=("battery_formula", "size"),
        n_basic_screening=("basic_screening_pass", "sum"),
        n_hard_exclusion_free=("hard_exclusion_free_candidate", "sum"),
        n_conservative=("conservative_earth_abundant_candidate", "sum"),
        n_exact_cde=("cde_exact_formula_match", "sum"),
        median_voltage=("average_voltage", "median"),
        median_capacity=("capacity_grav", "median"),
        median_energy=("energy_grav", "median"),
        median_mp_score=("mp_preliminary_score", "median"),
    )
    .reset_index()
    .sort_values("n_conservative", ascending=False)
)

assert int(family_summary["n_records"].sum()) == len(df_master_ml), (
    "ERROR: Family summary is inflated. Check duplicate rows."
)

family_summary_path = TABLE_DIR / "family_level_summary_FIXED.csv"
family_summary.to_csv(family_summary_path, index=False)

print("Saved:", family_summary_path)
print("Total family-summary records:", int(family_summary["n_records"].sum()))
print("Expected:", len(df_master_ml))

display(family_summary)


In [ ]:
# Fixed Cell 20 — Final output audit and row-count checks

final_outputs = [
    METRICS_DIR / "ml_grouped_cv_metrics_all_targets.csv",
    METRICS_DIR / "ml_best_model_by_target_protocol.csv",
    TABLE_DIR / "table_ml_grouped_cv_summary.csv",
    PRED_DIR / "ml_grouped_cv_oof_predictions_all_targets.csv",
    FEATURE_DIR / "protocol_feature_manifest.json",
    FEATURE_DIR / "extratrees_C_strict_feature_importance.csv",
    OUTPUT_DIR / "mp_na_electrodes_master_v2_with_ml_predictions_FIXED.csv",
    TABLE_DIR / "candidate_shortlist_all_records_FIXED.csv",
    TABLE_DIR / "candidate_shortlist_unique_for_manuscript_FIXED.csv",
    TABLE_DIR / "family_level_summary_FIXED.csv",
]

print("Final output file audit:")

for path in final_outputs:
    status = "FOUND" if path.exists() else "MISSING"
    size_mb = path.stat().st_size / (1024 * 1024) if path.exists() else 0
    print(f"{status:7s} | {size_mb:8.2f} MB | {path}")

print("\nCritical row-count checks:")
print("Original master rows:", len(df))
print("Fixed ML master rows:", len(df_master_ml))
print("All candidate shortlist rows:", len(df_candidate_shortlist))
print("Unique manuscript shortlist rows:", len(df_candidate_shortlist_unique))
print("Family-summary total rows:", int(family_summary["n_records"].sum()))

assert len(df_master_ml) == len(df), "ERROR: Fixed ML master row count is wrong."
assert int(family_summary["n_records"].sum()) == len(df_master_ml), "ERROR: Family summary row count is wrong."
assert len(df_candidate_shortlist) <= len(df_master_ml), "ERROR: Candidate shortlist is inflated."

print("\nAll fixed checks passed.")

print("\nMost important fixed outputs:")
print("1.", METRICS_DIR / "ml_best_model_by_target_protocol.csv")
print("2.", TABLE_DIR / "table_ml_grouped_cv_summary.csv")
print("3.", OUTPUT_DIR / "mp_na_electrodes_master_v2_with_ml_predictions_FIXED.csv")
print("4.", TABLE_DIR / "candidate_shortlist_all_records_FIXED.csv")
print("5.", TABLE_DIR / "candidate_shortlist_unique_for_manuscript_FIXED.csv")
print("6.", TABLE_DIR / "family_level_summary_FIXED.csv")
print("7.", FEATURE_DIR / "extratrees_C_strict_feature_importance.csv")
